# CONFLUENCE Tutorial - 3: Lumped Basin Workflow (East River at Almont)

## Introduction

## Lumped Basin Modeling Philosophy
Lumped basin modeling treats the entire watershed as one homogeneous computational unit, spatially averaging all variability across the catchment. This approach establishes baseline performance by determining whether a model can capture fundamental watershed response before adding spatial complexity, while its simplified parameter structure facilitates clear understanding of which parameters control model behavior and hydrological processes.

## Case Study: East River at Almont
Here, we focus on the East River at Almont watershed, located in the Sierra Nevada of Colorado, US. This watershed encompasses approximately *1,958* km² with elevations ranging from *3,500* m at the outlet to over *8,600* m in the headwaters, representing a snow-dominated mountain system with pronounced seasonal cycles. The basin is monitored by USGS station 09112500, which provides long-term streamflow observations essential for model validation.

## Step 1: Basin-Scale Workflow Setup
Building on the point-scale modeling expertise from Tutorials 01a and 01b, we now advance to basin-scale hydrological modeling. This represents a scaling transition from process validation at individual sites to integrated watershed simulation that captures the collective hydrological response of an entire catchment.

The same CONFLUENCE architecture seamlessly handles this transition, demonstrating the framework's scalability from point validation through basin-scale prediction while maintaining reproducible workflow principles.

In [ ]:
import os
# change to CONFLUENCE root directory
os.chdir('/home/dlhogan/projects/forked-repos/CONFLUENCE-uwmtnhydro')

In [ ]:
# Import the libraries we'll need in this notebook
import sys
from pathlib import Path
import yaml
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
from datetime import datetime
import xarray as xr
import numpy as np
import shutil

# Add CONFLUENCE to path
confluence_path = Path().resolve()
sys.path.append(str(confluence_path))

# Import main CONFLUENCE class
from CONFLUENCE import CONFLUENCE

# import custom utility functions
from utils.custom.calc import *
from utils.custom.plotting import *
from utils.custom.adjust_settings import *

# Set up plotting style
plt.style.use('default')
%matplotlib inline

In [ ]:
today = datetime.now().strftime('%Y%m%d')

In [ ]:
relevant_detail = 'spongy'  # Update this based on the specific model configuration (e.g., 'lumped', 'semi_distributed')
name = f'tuolumne_basin_lumped_{relevant_detail}_{today}'
# =============================================================================
# CONFIGURATION FOR SEMI-DISTRIBUTED TUOLUMNE RIVER MODELING
# =============================================================================

# Set directory paths
CONFLUENCE_CODE_DIR = confluence_path
CONFLUENCE_DATA_DIR = Path('/scratch/dlhogan/ess-project-data/') 

# Load template configuration and customize for semi-distributed modeling
config_template_path = CONFLUENCE_CODE_DIR / 'ess-project' / '0_config_files' / 'config_East.yaml'

with open(config_template_path, 'r') as f:
    config_dict = yaml.safe_load(f)

# Update for semi-distributed East River modeling
config_updates = {
    'CONFLUENCE_CODE_DIR': str(CONFLUENCE_CODE_DIR),
    'CONFLUENCE_DATA_DIR': str(CONFLUENCE_DATA_DIR),
    'DOMAIN_NAME': 'East_River_lumped',
    'EXPERIMENT_ID': f'{name}',  # Update experiment ID
    'DOMAIN_DEFINITION_METHOD': 'lumped',    # KEY CHANGE: watershed delineation vs lumped
    'STREAM_THRESHOLD': 5000,                   # Controls number of sub-basins
    'DOMAIN_DISCRETIZATION': 'GRUs',            # Grouped Response Units
    'SPATIAL_MODE': 'Lumped',               # Changed from 'Lumped'
    'HYDROLOGICAL_MODEL': 'SUMMA',
    'ROUTING_MODEL': 'mizuRoute',
    'EXPERIMENT_TIME_START': '2013-10-01 01:00',
    'EXPERIMENT_TIME_END':  '2022-09-30 23:00',
    'CALIBRATION_PERIOD': '2011-10-01, 2013-09-30',
    'EVALUATION_PERIOD': '2015-10-01, 2018-09-30',
    'SPINUP_PERIOD': '2010-10-01, 2013-09-30',
    'MPI_PROCESSES': 4,                       # Adjust based on available resources
    'TAUDEM_DIR': '/home/dlhogan/tools/src/TauDEM/bin',
    'DELINEATION_METHOD': 'TauDEM',  # Changed from 'TauDEM'
    'CATCHMENT_SHP_HRUID': 'HRU_ID',          # Updated HRU ID field
    'CATCHMENT_SHP_GRUID': 'GRU_ID',
    'SIM_REACH_ID': -1,                       # Changed to East River outlet reach ID
    'ROUTING_DELINEATION': 'lumped',        # Changed to 'lumped' or remove
    'APPLY_LAPSE_RATE': False,                 # Changed to True to apply lapse rate
    'OPTIMIZATION_METRIC': 'NSE',          # Changed to 'NSE' for emulation
    'ITERATIVE_OPTIMIZATION_ALGORITHM': 'DDS',  # Changed to 'DDS' for optimization
    'NUMBER_OF_ITERATIONS': 200,
    'BASIN_PARAMS_TO_CALIBRATE': 'routingGammaScale, routingGammaShape',
    'SUMMA_RUNOFF_VAR': 'scalarTotalRunoff'    # Use scalarTotalRunoff for calibration (not averageRoutedRunoff)
}

config_dict.update(config_updates)

# Save configuration
temp_config_path = CONFLUENCE_CODE_DIR / 'ess-project' / '0_config_files' / 'config_East_lumped_v1.yaml'
with open(temp_config_path, 'w') as f:
    yaml.dump(config_dict, f, default_flow_style=False, sort_keys=False)

# Initialize CONFLUENCE
confluence = CONFLUENCE(temp_config_path)

# Display configuration
print("=== Directory Configuration ===")
print(f"Code Directory: {CONFLUENCE_CODE_DIR}")
print(f"Data Directory: {CONFLUENCE_DATA_DIR}")
print("\n=== Key Configuration Settings ===")
print(f"Domain Name: {confluence.config['DOMAIN_NAME']}")
print(f"Pour Point: {confluence.config['POUR_POINT_COORDS']}")
print(f"Spatial Mode: {confluence.config['SPATIAL_MODE']}")
print(f"Model: {confluence.config['HYDROLOGICAL_MODEL']}")
print(f"Simulation Period: {confluence.config['EXPERIMENT_TIME_START']} to {confluence.config['EXPERIMENT_TIME_END']}")
print(f"Basin Parameters to Calibrate: {confluence.config['BASIN_PARAMS_TO_CALIBRATE']}")


### Model Decision & Parameter Changes

In [ ]:
from utils.custom.adjust_settings import edit_localParamInfo, edit_modelDecisions
modelDecision_file = Path("/home/dlhogan/projects/forked-repos/CONFLUENCE-uwmtnhydro/ess-project/0_base_settings/SUMMA/modelDecisions.txt")
localParamInfo_file = Path("/home/dlhogan/projects/forked-repos/CONFLUENCE-uwmtnhydro/ess-project/0_base_settings/SUMMA/localParamInfo.txt")
basinParamInfo_file = Path("/home/dlhogan/projects/forked-repos/CONFLUENCE-uwmtnhydro/ess-project/0_base_settings/SUMMA/basinParamInfo.txt")

# Model Decision changes
modelDecision_updates = {
    'groundwatr': 'bigBuckt',  # 'bigBuckt', could use 'noXplict'
    'bcLowrSoiH': 'drainage', # change from 'drainage' to 'presHead'
    'spatial_gw': 'localColumn', # 'localColumn =
    'alb_method': 'varDecay' # change from 'varDecay' to 'conDecay'
    }

# Parameter changes - ensure you have 4 characters after the decimal
localParamInfo_updates = {
    'tempCritRain': '274.1000',
    'upperBoundtemp': '272.1600',
    'lowerBoundtemp': '274.1600',
    'albedoMinVisible': '0.3000',
    'albedoDecayRate' : '1.0d+6',
    'k_soil': '1.0d-5',  # ↑ INCREASE from 1.0d-5 (less restrictive)
    'theta_sat': '0.1500',  # ↑ INCREASE from 0.600 (more realistic)
    'theta_res': '0.0750',  # ↓ DECREASE from 0.075 (less restrictive)
    "aquiferScaleFactor": "0.5000",  # back toward default
}

# Basin Parameter changes
basinParamInfo_updates = {
    "basin__aquiferHydCond": "1.0000",     # bump up so bucket actually drains
    "basin__aquiferScaleFactor": "0.5000",  # back toward default
    # "basin__aquiferBaseflowExp": "3.0",
    "routingGammaShape": "2.5000",  # ↑ Increase from 2.5
    "routingGammaScale": "4.6d+4",  # ↓ Reduce from 4.6d+6
}

edit_modelDecisions(modelDecision_file, modelDecision_updates)
edit_localParamInfo(localParamInfo_file, localParamInfo_updates)
edit_localParamInfo(basinParamInfo_file, basinParamInfo_updates)

# Sync updated settings into the scratch project settings directory
project_settings_dir = Path(config_dict['CONFLUENCE_DATA_DIR']) / f"domain_{config_dict['DOMAIN_NAME']}" / "settings" / "SUMMA"
project_settings_dir.mkdir(parents=True, exist_ok=True)
shutil.copy(localParamInfo_file, project_settings_dir / "localParamInfo.txt")
shutil.copy(modelDecision_file, project_settings_dir / "modelDecisions.txt")
shutil.copy(basinParamInfo_file, project_settings_dir / "basinParamInfo.txt")
print(f"Synced SUMMA settings to: {project_settings_dir}")

In [ ]:
# Project Initialization
project_dir = confluence.managers['project'].setup_project()

# Create pour point
pour_point_path = confluence.managers['project'].create_pour_point()

# List created directories
print("\nCreated directories:")
for item in sorted(project_dir.iterdir()):
    if item.is_dir():
        print(f"  📁 {item.name}")

print("\nDirectory purposes:")
print("  📁 shapefiles: Domain geometry (watershed, pour points, river network)")
print("  📁 attributes: Static characteristics (elevation, soil, land cover)")
print("  📁 forcing: Meteorological inputs (precipitation, temperature)")
print("  📁 simulations: Model outputs")
print("  📁 evaluation: Performance metrics and comparisons")
print("  📁 plots: Visualizations")
print("  📁 optimisation: Calibration results")

## Step 2: Basin Representation and Spatial Discretization Fundamentals
The transition from point-scale to basin-scale modeling requires fundamental decisions about how to represent spatial heterogeneity within the watershed. Unlike point-scale modeling where we assume uniform conditions, basin-scale modeling must address the challenge of capturing spatial variability.

### Scientific Context: Basin Representation Philosophy
Basin-scale hydrological modeling confronts significant spatial heterogeneity challenges that profoundly influence water cycling processes. 
- Elevation gradients create complex patterns in temperature lapse rates, precipitation distributions, and snow line dynamics that control seasonal water storage and release. 
- Vegetation patterns ranging from dense forest to alpine zones fundamentally alter evapotranspiration rates and interception processes. 
- Soil variability across the landscape affects infiltration capacity, water storage potential, and drainage characteristics that determine runoff generation mechanisms. 
- Topographic effects including slope, aspect, and drainage network configuration control flow routing and energy balance processes. 
- Climate gradients manifest through orographic precipitation enhancement, temperature inversions, and wind pattern modifications that create substantial spatial variability in hydrological drivers.

### Representation Strategies and Discretization Approaches
Hydrological modeling addresses these spatial complexities through three primary representation strategies, each offering distinct advantages and computational trade-offs. 
- The lumped approach treats the entire watershed as a single computational unit with spatially-averaged characteristics, providing computational efficiency and parameter interpretability while sacrificing spatial detail. 
- Semi-distributed strategies create multiple computational units based on landscape similarity criteria such as elevation bands, soil types, or land cover classes, balancing spatial representation with computational tractability. 
- Fully distributed approaches employ grid-based representation with explicit spatial patterns, capturing detailed heterogeneity at the cost of increased computational demands and parameter complexity.

The Grouped Response Unit (GRU) concept provides flexible spatial discretization that allows users to choose the appropriate level of complexity for their scientific objectives and computational constraints. This framework enables seamless transitions from lumped representations through increasingly detailed semi-distributed configurations, ensuring that model complexity aligns with both scientific questions and available computational resources.

In [ ]:
# Execute attribute acquisition
# confluence.managers['data'].acquire_attributes()
print("✅ Basin-scale attribute acquisition complete")

In [ ]:
# Execute watershed delineation
watershed_path = confluence.managers['domain'].define_domain()
print("✅ Watershed delineation complete")

# Execute domain discretization
confluence.managers['domain'].discretize_domain()
print("✅ Domain discretization complete")


## VISUALIZATION AND ANALYSIS OF BASIN REPRESENTATION

In [ ]:
# Load spatial data
hru_path = str(Path(config_dict['CONFLUENCE_DATA_DIR']) / f"domain_{config_dict['DOMAIN_NAME']}" / 'shapefiles' / 'catchment' / f"{config_dict['DOMAIN_NAME']}_HRUs_{config_dict['DOMAIN_DISCRETIZATION']}.shp")
watershed_gdf = gpd.read_file(str(watershed_path[1]))
hru_gdf = gpd.read_file(hru_path)
pour_point_gdf = gpd.read_file(pour_point_path)

# Project to appropriate CRS for area calculations
# For East River at Almont,, UTM Zone 13N is appropriate
target_crs = 'EPSG:32613'  # UTM Zone 13N
watershed_projected = watershed_gdf.to_crs(target_crs)
hru_projected = hru_gdf.to_crs(target_crs)

print(f"\n📋 Basin Characteristics:")
total_area_m2 = watershed_projected.geometry.area.sum()
total_area_km2 = total_area_m2 / 1e6
print(f"   Watershed area: {total_area_km2:.1f} km²")
print(f"   Number of GRUs: {len(hru_gdf)}")
print(f"   Average GRU size: {total_area_km2/len(hru_gdf):.1f} km²")

# Display HRU characteristics if available
if 'elevation' in hru_gdf.columns:
    print(f"   Elevation range: {hru_gdf['elevation'].min():.0f}m to {hru_gdf['elevation'].max():.0f}m")
if 'slope' in hru_gdf.columns:
    print(f"   Slope range: {hru_gdf['slope'].min():.1f}° to {hru_gdf['slope'].max():.1f}°")

# Create comprehensive visualization
print(f"\n🗺️  Creating basin representation visualization...")
fig, axes = plt.subplots(1, 2, figsize=(8, 4))

# Left plot: Watershed boundary and pour point (use original CRS for plotting)
ax1 = axes[0]
watershed_gdf.plot(ax=ax1, facecolor='lightblue', edgecolor='navy', 
                  linewidth=2, alpha=0.7)
pour_point_gdf.plot(ax=ax1, color='red', markersize=100, marker='o',
                   edgecolor='white', linewidth=2, zorder=5)
ax1.set_title('Delineated Watershed Boundary', fontsize=14, fontweight='bold')
ax1.set_xlabel('Longitude', fontsize=12)
ax1.set_ylabel('Latitude', fontsize=12)
ax1.grid(True, alpha=0.3)

# Add area annotation
ax1.text(0.02, 0.98, f'Area: {total_area_km2:.1f} km²\nPour Point: USGS {config_dict["STATION_ID"]}',
         transform=ax1.transAxes, fontsize=10, verticalalignment='top',
         bbox=dict(facecolor='white', alpha=0.8, boxstyle='round,pad=0.3'))

# Right plot: HRU discretization
ax2 = axes[1]
# Color HRUs by elevation if available, otherwise by area (using projected areas)
if 'elev_mean' in hru_gdf.columns:
    hru_gdf.plot(ax=ax2, column='elev_mean', cmap='terrain', 
                edgecolor='black', linewidth=0.5, legend=True)
    colorbar_label = 'Elevation (m)'
else:
    # Use projected geometries for area calculation
    hru_areas_km2 = hru_projected.geometry.area / 1e6
    hru_gdf.plot(ax=ax2, column=hru_areas_km2, cmap='viridis',
                edgecolor='black', linewidth=0.5, legend=True) 
    colorbar_label = 'Area (km²)'

pour_point_gdf.plot(ax=ax2, color='red', markersize=100, marker='o',
                   edgecolor='white', linewidth=2, zorder=5)
ax2.set_title(f'GRU Discretization ({len(hru_gdf)} units)', fontsize=14, fontweight='bold')
ax2.set_xlabel('Longitude', fontsize=12)
ax2.set_ylabel('Latitude', fontsize=12)
ax2.grid(True, alpha=0.3)

# Add discretization info
ax2.text(0.02, 0.98, f'GRUs: {len(hru_gdf)}\nAvg. size: {total_area_km2/len(hru_gdf):.1f} km²',
         transform=ax2.transAxes, fontsize=10, verticalalignment='top',
         bbox=dict(facecolor='white', alpha=0.8, boxstyle='round,pad=0.3'))

plt.suptitle(f'East River at Almont: Basin Representation', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## Step 3: Data Pipeline for Basin-Scale Streamflow Modeling
The same model-agnostic preprocessing framework now scales from point validation to basin-scale streamflow simulation. The core philosophy remains unchanged—standardized, quality-controlled data products—but the spatial context shifts from single locations to integrated watershed responses

In [ ]:
# =============================================================================
# FORCING DATA PROCESSOR - Using reusable module
# =============================================================================
from utils.custom.forcing_processor import ForcingProcessor

# Initialize the forcing processor with config
forcing_processor = ForcingProcessor(config_dict)

# Check current status of forcing data
status = forcing_processor.check_status()

In [ ]:
# =============================================================================
# STEP 3a: MERGE RAW ERA5 DATA (Surface + Pressure Level)
# =============================================================================
# Combines ERA5_surface_YYYYMM.nc + ERA5_pressureLevel137_YYYYMM.nc into ERA5_merged_YYYYMM.nc

forcing_processor.merge_era5()

In [ ]:
# =============================================================================
# STEP 3b: BASIN-AVERAGE MERGED DATA OVER CATCHMENT
# =============================================================================
# Uses CONFLUENCE's built-in EASYMORE resampler for spatial averaging

forcing_processor.basin_average(confluence=confluence)

In [ ]:
# =============================================================================
# STEP 3c: CREATE SUMMA INPUT (with optional adjustments)
# =============================================================================
# Optional: apply temperature offset and/or precipitation scaling

forcing_processor.create_summa_input(
    temp_adjustment=0.0,      # Kelvin offset (0 = no change)
    precip_multiplier=1.0,     # Scaling factor (1.0 = no change)
    recalc_longwave=True
)

# Verify final status
forcing_processor.check_status()

### Observed data

In [ ]:
# Execute streamflow data processing
confluence.managers['data'].process_observed_data()
print("✅ Streamflow data processing complete")

# NOTE: Forcing acquisition/processing is handled in cells above (Step 3a-3c)
# confluence.managers['data'].acquire_forcings()  # Skipped - forcing already processed
print("✅ Forcing data prepared via manual processing above")

### Model Agnostic preprocessing

In [ ]:
# Execute model-agnostic preprocessing
confluence.managers['data'].run_model_agnostic_preprocessing()
print("✅ Model-agnostic preprocessing complete")

In [ ]:
# Execute model-specific preprocessing
confluence.managers['model'].preprocess_models()
print("✅ Basin-scale model configuration complete")

#### No longer really needed.
This is the manual update to the forcing files

In [ ]:
# open the dem 
dem_path = str(Path(config_dict['CONFLUENCE_DATA_DIR']) / f"domain_{config_dict['DOMAIN_NAME']}" / 'attributes' / 'elevation' / "dem" / "domain_East_River_lumped_elv.tif")

# using rasterio open the dem
import rasterio
with rasterio.open(dem_path) as dem_src:
    dem_data = dem_src.read(1)  # Read the first band
    dem_transform = dem_src.transform
    dem_crs = dem_src.crs

# calculate the mean elevation
mean_elevation = np.nanmean(dem_data)

# adjust temperature by difference between mean dem elevation and mean hru elevation
mean_hru_elevation = hru_gdf['elev_mean'].mean()
elevation_difference = mean_hru_elevation - mean_elevation
lapse_rate = 6.5 / 1000  # °C per meter
temperature_adjustment = elevation_difference * lapse_rate

# winter adjustment (Jan - May)
monthly_adjustment = {
    1: 1.0,  # January
    2: 1.0,  # February
    3: 1.0,  # March
    4: 1.0,  # April
    5: 1.0,  # May
    6: 0.0,  # June
    7: -1.0,  # July
    8: -1.0,  # August
    9: -1.0,  # September
    10: -1.0, # October
    11: -1.0, # November
    12: 0.0  # December
}

In [ ]:
file_list = list(Path(config_dict['CONFLUENCE_DATA_DIR']).glob(f"domain_{config_dict['DOMAIN_NAME']}/forcing/forcing_noTadjust/{config_dict['DOMAIN_NAME']}_ERA5_remapped_ERA5_merged_*.nc"))
file_list = sorted(file_list)
for file in file_list:
    ds = xr.open_dataset(str(file))
    # adjust temperature
    ds['airtemp'] = ds['airtemp'] + temperature_adjustment
    # determe additional monthly temperature adjustment
    # monthlyTempAdjustment = monthly_adjustment[ds['airtemp'].time.dt.month.values[0]]
    # ds['airtemp'] = ds['airtemp'] + monthlyTempAdjustment
    # adjust precipitation
    # ds['pptrate'] = ds['pptrate'] * 0.90  # Example: decrease precipitation by 10%
    # recalculate incoming longwave radiation using empirical formula
    ds['LWRadAtm'] = empirical_lw_dilley_obrien(ds['airtemp'], ds['spechum'], ds['airpres'])
    # save the modified dataset back to new file
    adjusted_file_path = str(file).replace("forcing_noTadjust", "SUMMA_input")
    ds.to_netcdf(adjusted_file_path)
    ds.close()

## Step 4: Streamlined Basin-Scale Model Execution
The same SUMMA process-based physics now scales from point validation to integrated basin simulation, but with the critical addition of streamflow routing. This represents a fundamental modeling advancement: from isolated vertical processes to coupled vertical-horizontal water transport that generates streamflow at the basin outlet.

#### Step 4b: Model Calibration Optimization

In [ ]:
confluence.managers['optimization'].calibrate_model()

### Analyzing Calibration Results & Parameter Sensitivity
After calibration completes, we analyze:
1. **Best Parameters** - The optimal parameter set found
2. **Convergence History** - How performance improved over iterations
3. **Parameter Sensitivity** - Which parameters most influence model performance

In [ ]:
# =============================================================================
# LOAD AND ANALYZE OPTIMIZATION RESULTS
# =============================================================================

# Define paths to optimization outputs
opt_base_dir = project_dir / "optimisation"
experiment_id = config_dict['EXPERIMENT_ID']

# Find the most recent optimization results folders
dds_folders = sorted(opt_base_dir.glob("dds_*"))
history_files = sorted(opt_base_dir.glob("*_dds_history.csv"))

print("=" * 60)
print("CALIBRATION RESULTS ANALYSIS")
print("=" * 60)

# Load the DDS history (all iterations with parameters)
if history_files:
    latest_history_file = history_files[-1]
    dds_history = pd.read_csv(latest_history_file)
    print(f"\n📁 Loaded history from: {latest_history_file.name}")
    print(f"   Total iterations: {len(dds_history)}")
else:
    print("⚠️ No DDS history file found")
    dds_history = None

# Load best parameters from the optimization folder
if dds_folders:
    latest_dds_folder = dds_folders[-1]
    best_params_file = latest_dds_folder / "best_parameters.csv"
    opt_history_file = latest_dds_folder / "optimization_history.csv"
    
    if best_params_file.exists():
        best_params_df = pd.read_csv(best_params_file)
        print(f"\n✅ BEST PARAMETERS FOUND:")
        print("-" * 40)
        for _, row in best_params_df.iterrows():
            print(f"   {row['parameter']:20s}: {row['value']:.6g}")
        
    if opt_history_file.exists():
        opt_history_df = pd.read_csv(opt_history_file)
        best_score = opt_history_df['best_score'].max()
        best_gen = opt_history_df.loc[opt_history_df['best_score'].idxmax(), 'generation']
        print(f"\n📊 Best {config_dict.get('OPTIMIZATION_METRIC', 'NSE')}: {best_score:.4f} (Generation {best_gen})")

# Get the parameter columns (exclude iteration and metric)
if dds_history is not None:
    metric_col = config_dict.get('OPTIMIZATION_METRIC', 'NSE')
    param_cols = [c for c in dds_history.columns if c not in ['iteration', metric_col]]
    
    print(f"\n📋 Parameters calibrated: {len(param_cols)}")
    for p in param_cols:
        print(f"   - {p}")

In [ ]:
# =============================================================================
# VISUALIZATION 1: OPTIMIZATION CONVERGENCE
# =============================================================================

if dds_history is not None and len(dds_history) > 1:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    metric_col = config_dict.get('OPTIMIZATION_METRIC', 'NSE')
    
    # 1. Convergence plot - all iterations
    ax1 = axes[0, 0]
    ax1.scatter(range(len(dds_history)), dds_history[metric_col], alpha=0.6, c='steelblue', s=30)
    
    # Running best line
    running_best = dds_history[metric_col].cummax()
    ax1.plot(range(len(dds_history)), running_best, 'r-', linewidth=2, label='Running Best')
    
    # Mark overall best
    best_idx = dds_history[metric_col].idxmax()
    best_val = dds_history[metric_col].max()
    ax1.scatter([best_idx], [best_val], c='gold', s=150, marker='*', 
                edgecolor='black', zorder=5, label=f'Best: {best_val:.4f}')
    
    ax1.set_xlabel('Iteration', fontsize=11)
    ax1.set_ylabel(f'{metric_col}', fontsize=11)
    ax1.set_title(f'Optimization Convergence ({len(dds_history)} iterations)', fontweight='bold')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 2. Score distribution histogram
    ax2 = axes[0, 1]
    valid_scores = dds_history[metric_col][dds_history[metric_col] > -1]  # Filter invalid
    ax2.hist(valid_scores, bins=20, color='steelblue', edgecolor='black', alpha=0.7)
    ax2.axvline(best_val, color='red', linestyle='--', linewidth=2, label=f'Best: {best_val:.4f}')
    ax2.axvline(valid_scores.mean(), color='orange', linestyle='--', linewidth=2, label=f'Mean: {valid_scores.mean():.4f}')
    ax2.set_xlabel(f'{metric_col}', fontsize=11)
    ax2.set_ylabel('Count', fontsize=11)
    ax2.set_title(f'{metric_col} Distribution Across Iterations', fontweight='bold')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # 3. Parameter correlation with metric - top 4 most variable params
    ax3 = axes[1, 0]
    param_std = dds_history[param_cols].std()
    top_params = param_std.nlargest(4).index.tolist()
    
    colors = plt.cm.tab10(np.linspace(0, 1, len(top_params)))
    for i, param in enumerate(top_params):
        # Normalize parameter for comparison
        param_norm = (dds_history[param] - dds_history[param].min()) / (dds_history[param].max() - dds_history[param].min() + 1e-10)
        ax3.scatter(param_norm, dds_history[metric_col], alpha=0.4, label=param, color=colors[i], s=20)
    
    ax3.set_xlabel('Normalized Parameter Value', fontsize=11)
    ax3.set_ylabel(f'{metric_col}', fontsize=11)
    ax3.set_title('Parameter Values vs Performance\n(Top 4 Most Variable)', fontweight='bold')
    ax3.legend(fontsize=8, loc='lower right')
    ax3.grid(True, alpha=0.3)
    
    # 4. Best parameters bar chart
    ax4 = axes[1, 1]
    if 'best_params_df' in dir():
        # Normalize values for visualization (log scale for widely varying params)
        params = best_params_df['parameter'].values
        values = best_params_df['value'].values
        
        # Use log scale for visualization
        log_values = np.log10(np.abs(values) + 1e-10)
        bars = ax4.barh(params, log_values, color='steelblue', edgecolor='black')
        ax4.set_xlabel('log₁₀(|value|)', fontsize=11)
        ax4.set_title('Best Parameter Values (log scale)', fontweight='bold')
        ax4.grid(True, alpha=0.3, axis='x')
    
    plt.suptitle(f'DDS Optimization Analysis - {config_dict["DOMAIN_NAME"]}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ Not enough iterations for convergence visualization")

In [ ]:
# =============================================================================
# VISUALIZATION 2: PARAMETER SENSITIVITY ANALYSIS
# =============================================================================

if dds_history is not None and len(dds_history) > 5:
    metric_col = config_dict.get('OPTIMIZATION_METRIC', 'NSE')
    
    # Calculate sensitivity metrics for each parameter
    sensitivity_results = []
    
    for param in param_cols:
        param_values = dds_history[param]
        metric_values = dds_history[metric_col]
        
        # Filter valid runs (NSE > -1)
        valid_mask = metric_values > -1
        # if valid_mask.sum() < 3:
        #     continue
            
        param_valid = param_values[valid_mask]
        metric_valid = metric_values[valid_mask]
        
        # Calculate correlation between parameter and metric
        corr = param_valid.corr(metric_valid)
        
        # Calculate range-based sensitivity
        param_range = param_valid.max() - param_valid.min()
        param_cv = param_valid.std() / (param_valid.mean() + 1e-10)  # Coefficient of variation
        
        # Split parameter into bins and calculate metric statistics
        try:
            bins = pd.qcut(param_valid, q=3, labels=['Low', 'Mid', 'High'], duplicates='drop')
            metric_by_bin = metric_valid.groupby(bins).mean()
            metric_spread = metric_by_bin.max() - metric_by_bin.min()
        except:
            metric_spread = 0
        
        sensitivity_results.append({
            'parameter': param,
            'correlation': 0,#abs(corr) if not np.isnan(corr) else 0,
            'metric_spread': metric_spread,
            'param_cv': param_cv,
            'best_value': dds_history.loc[dds_history[metric_col].idxmax(), param]
        })
    
    sens_df = pd.DataFrame(sensitivity_results)
    sens_df = sens_df.sort_values('correlation', ascending=False)
    
    # Create sensitivity visualization
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    
    # 1. Correlation-based sensitivity ranking
    ax1 = axes[0, 0]
    colors = plt.cm.RdYlGn(sens_df['correlation'].values)
    bars = ax1.barh(sens_df['parameter'], sens_df['correlation'], color=colors, edgecolor='black')
    ax1.set_xlabel(f'|Correlation with {metric_col}|', fontsize=11)
    ax1.set_title('Parameter Sensitivity Ranking\n(by correlation with metric)', fontweight='bold')
    ax1.set_xlim(0, 1)
    ax1.grid(True, alpha=0.3, axis='x')
    
    # Add correlation values as text
    for i, (idx, row) in enumerate(sens_df.iterrows()):
        ax1.text(row['correlation'] + 0.02, i, f'{row["correlation"]:.3f}', va='center', fontsize=9)
    
    # 2. Parameter vs Metric scatter plots (top 6 sensitive parameters)
    ax2 = axes[0, 1]
    top_sensitive = sens_df.head(6)['parameter'].tolist()
    
    for i, param in enumerate(top_sensitive[:3]):  # Top 3 most sensitive
        param_norm = (dds_history[param] - dds_history[param].min()) / (dds_history[param].max() - dds_history[param].min() + 1e-10)
        ax2.scatter(param_norm, dds_history[metric_col], alpha=0.5, label=param, s=25)
    
    ax2.set_xlabel('Normalized Parameter Value', fontsize=11)
    ax2.set_ylabel(metric_col, fontsize=11)
    ax2.set_title('Top 3 Most Sensitive Parameters', fontweight='bold')
    ax2.legend(fontsize=9)
    ax2.grid(True, alpha=0.3)
    
    # 3. Parameter exploration (how much each parameter was varied)
    ax3 = axes[1, 0]
    exploration = []
    for param in param_cols:
        param_data = dds_history[param]
        exploration.append({
            'parameter': param,
            'explored_range': (param_data.max() - param_data.min()) / (param_data.mean() + 1e-10)
        })
    explore_df = pd.DataFrame(exploration).sort_values('explored_range', ascending=True)
    
    ax3.barh(explore_df['parameter'], explore_df['explored_range'], color='coral', edgecolor='black')
    ax3.set_xlabel('Relative Range Explored (normalized)', fontsize=11)
    ax3.set_title('Parameter Search Space Exploration', fontweight='bold')
    ax3.grid(True, alpha=0.3, axis='x')
    
    # 4. Parallel coordinates for top runs
    ax4 = axes[1, 1]
    # Get top 10% of runs
    n_top = max(int(len(dds_history) * 0.1), 3)
    top_runs = dds_history.nlargest(n_top, metric_col)
    
    # Normalize parameters for parallel coordinates
    param_normalized = pd.DataFrame()
    for param in param_cols[:8]:  # Limit to 8 params for readability
        param_normalized[param] = (top_runs[param] - dds_history[param].min()) / (dds_history[param].max() - dds_history[param].min() + 1e-10)
    
    # Plot each top run as a line
    x_coords = range(len(param_normalized.columns))
    for idx, row in param_normalized.iterrows():
        score = top_runs.loc[idx, metric_col]
        alpha = 0.3 + 0.7 * (score - top_runs[metric_col].min()) / (top_runs[metric_col].max() - top_runs[metric_col].min() + 1e-10)
        ax4.plot(x_coords, row.values, alpha=alpha, linewidth=1.5, 
                color=plt.cm.viridis((score - top_runs[metric_col].min()) / (top_runs[metric_col].max() - top_runs[metric_col].min() + 1e-10)))
    
    ax4.set_xticks(x_coords)
    ax4.set_xticklabels(param_normalized.columns, rotation=45, ha='right', fontsize=8)
    ax4.set_ylabel('Normalized Value', fontsize=11)
    ax4.set_title(f'Parallel Coordinates: Top {n_top} Runs\n(brighter = better {metric_col})', fontweight='bold')
    ax4.set_ylim(-0.1, 1.1)
    ax4.grid(True, alpha=0.3)
    
    plt.suptitle(f'Parameter Sensitivity Analysis - {config_dict["DOMAIN_NAME"]}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Print summary
    print("\n" + "=" * 60)
    print("PARAMETER SENSITIVITY SUMMARY")
    print("=" * 60)
    print(f"\nMost Sensitive Parameters (ranked by |correlation| with {metric_col}):")
    for i, row in sens_df.head(5).iterrows():
        print(f"  {row['parameter']:20s}: r = {row['correlation']:.3f}, best value = {row['best_value']:.4g}")
    
    print(f"\nLeast Sensitive Parameters:")
    for i, row in sens_df.tail(3).iterrows():
        print(f"  {row['parameter']:20s}: r = {row['correlation']:.3f}")

else:
    print("⚠️ Not enough data for sensitivity analysis (need >5 iterations)")

In [ ]:
# =============================================================================
# LOAD BEST MODEL SIMULATION RESULTS
# =============================================================================

# Find the final/best simulation output from the DDS optimization
dds_summa_dir = project_dir / "simulations" / "run_dds" / "SUMMA"
dds_routing_dir = project_dir / "simulations" / "run_dds" / "mizuRoute"

# Look for the final simulation file (uses best parameters)
final_files = list(dds_summa_dir.glob("run_dds_final_*.nc"))
opt_files = list(dds_summa_dir.glob("run_dds_opt_*.nc"))

print("=" * 60)
print("BEST MODEL SIMULATION FILES")
print("=" * 60)

# The final file is the full simulation with best parameters
if final_files:
    best_summa_file = sorted(final_files)[-1]  # Most recent
    print(f"\n✅ Final simulation (best params, full period):")
    print(f"   {best_summa_file.name}")
    
    # Load the best simulation
    best_sim_ds = xr.open_dataset(best_summa_file)
    print(f"\n   Time range: {pd.Timestamp(best_sim_ds.time.values[0])} to {pd.Timestamp(best_sim_ds.time.values[-1])}")
    print(f"   Timesteps: {len(best_sim_ds.time)}")
    
else:
    print("⚠️ No final simulation file found")
    # Fall back to the optimization file
    if opt_files:
        best_summa_file = sorted(opt_files)[-1]
        print(f"\n📁 Using optimization simulation:")
        print(f"   {best_summa_file.name}")
        best_sim_ds = xr.open_dataset(best_summa_file)
    else:
        best_sim_ds = None

# Check for routing output
if dds_routing_dir.exists():
    routing_files = list(dds_routing_dir.glob("*.nc"))
    if routing_files:
        print(f"\n📁 Routing output available: {routing_files[0].name}")

# Load the best trial parameters
best_params_nc = project_dir / "simulations" / "run_dds" / "settings" / "SUMMA" / "trialParams.nc"
if best_params_nc.exists():
    print(f"\n✅ Best parameters file: {best_params_nc}")
    params_ds = xr.open_dataset(best_params_nc)
    print("\n   Calibrated parameter values in trialParams.nc:")
    for var in params_ds.data_vars:
        values = params_ds[var].values
        if values.size == 1:
            print(f"   {var:25s}: {values.item():.6g}")
        else:
            print(f"   {var:25s}: mean={np.mean(values):.6g}")
    params_ds.close()

In [ ]:
# =============================================================================
# EVALUATE BEST MODEL AGAINST OBSERVATIONS
# =============================================================================

if best_sim_ds is not None:
    # Load observations
    obs_path = project_dir / "observations" / "streamflow" / "preprocessed" / f"{config_dict['DOMAIN_NAME']}_streamflow_processed.csv"
    obs_df = pd.read_csv(obs_path, parse_dates=['datetime'])
    obs_df.set_index('datetime', inplace=True)
    
    # Extract simulated streamflow
    shp_file = gpd.read_file(str(project_dir / "shapefiles" / "catchment" / f"{config_dict['DOMAIN_NAME']}_HRUs_{config_dict['DOMAIN_DISCRETIZATION']}.shp"))
    shp_area = shp_file['GRU_area'].values[0] if 'GRU_area' in shp_file.columns else 1.0
    
    sim_q = best_sim_ds['averageRoutedRunoff'] * shp_area
    sim_df = sim_q.to_pandas()
    
    # Get calibration period
    cal_period = config_dict.get('CALIBRATION_PERIOD', '2011-10-01, 2013-09-30')
    cal_start, cal_end = [s.strip() for s in cal_period.split(',')]
    
    # Align to common period
    common_start = max(obs_df.index.min(), sim_df.index.min())
    common_end = min(obs_df.index.max(), sim_df.index.max())
    
    # Calculate metrics for different periods
    def calc_metrics(obs, sim):
        """Calculate NSE and KGE"""
        valid = ~(obs.isna() | sim.isna())
        o, s = obs[valid], sim[valid]
        
        # NSE
        nse = 1 - ((o - s) ** 2).sum() / ((o - o.mean()) ** 2).sum()
        
        # KGE
        r = o.corr(s)
        alpha = s.std() / o.std()
        beta = s.mean() / o.mean()
        kge = 1 - np.sqrt((r - 1)**2 + (alpha - 1)**2 + (beta - 1)**2)
        
        return {'NSE': nse, 'KGE': kge, 'r': r, 'alpha': alpha, 'beta': beta}
    
    # Calibration period metrics
    obs_cal = obs_df['discharge_cms'].loc[cal_start:cal_end]
    sim_cal = sim_df.loc[cal_start:cal_end]
    if isinstance(sim_cal, pd.DataFrame):
        sim_cal = sim_cal.iloc[:, 0]
    metrics_cal = calc_metrics(obs_cal, sim_cal)
    
    # Full period metrics
    obs_full = obs_df['discharge_cms'].loc[common_start:common_end]
    sim_full = sim_df.loc[common_start:common_end]
    if isinstance(sim_full, pd.DataFrame):
        sim_full = sim_full.iloc[:, 0]
    metrics_full = calc_metrics(obs_full, sim_full)
    
    print("=" * 60)
    print("BEST MODEL PERFORMANCE")
    print("=" * 60)
    print(f"\n📊 Calibration Period ({cal_start} to {cal_end}):")
    print(f"   NSE: {metrics_cal['NSE']:.3f}")
    print(f"   KGE: {metrics_cal['KGE']:.3f}")
    print(f"   r:   {metrics_cal['r']:.3f}")
    
    print(f"\n📊 Full Period ({common_start.date()} to {common_end.date()}):")
    print(f"   NSE: {metrics_full['NSE']:.3f}")
    print(f"   KGE: {metrics_full['KGE']:.3f}")
    print(f"   r:   {metrics_full['r']:.3f}")
    
    # Create comparison plot
    fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
    
    # Calibration period
    ax1 = axes[0]
    obs_cal_daily = obs_cal.resample('D').mean()
    sim_cal_daily = sim_cal.resample('D').mean()
    ax1.plot(obs_cal_daily.index, obs_cal_daily.values, 'b-', label='Observed', linewidth=1.5)
    ax1.plot(sim_cal_daily.index, sim_cal_daily.values, 'r-', label='Simulated (Best)', linewidth=1.5, alpha=0.8)
    ax1.set_ylabel('Discharge (m³/s)', fontsize=11)
    ax1.set_title(f'Calibration Period: NSE={metrics_cal["NSE"]:.3f}, KGE={metrics_cal["KGE"]:.3f}', fontweight='bold')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax1.axvspan(pd.Timestamp(cal_start), pd.Timestamp(cal_end), alpha=0.1, color='green', label='Cal Period')
    
    # Full simulation
    ax2 = axes[1]
    obs_full_daily = obs_full.resample('D').mean()
    sim_full_daily = sim_full.resample('D').mean()
    ax2.plot(obs_full_daily.index, obs_full_daily.values, 'b-', label='Observed', linewidth=1.5)
    ax2.plot(sim_full_daily.index, sim_full_daily.values, 'r-', label='Simulated (Best)', linewidth=1.5, alpha=0.8)
    ax2.set_ylabel('Discharge (m³/s)', fontsize=11)
    ax2.set_xlabel('Date', fontsize=11)
    ax2.set_title(f'Full Period: NSE={metrics_full["NSE"]:.3f}, KGE={metrics_full["KGE"]:.3f}', fontweight='bold')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # Shade calibration period
    ax2.axvspan(pd.Timestamp(cal_start), pd.Timestamp(cal_end), alpha=0.15, color='green')
    
    plt.suptitle(f'Best Model Validation - {config_dict["DOMAIN_NAME"]}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    best_sim_ds.close()
else:
    print("⚠️ No best simulation data available")

In [ ]:
# Define paths
project_dir = Path(config_dict['CONFLUENCE_DATA_DIR']) / f"domain_{config_dict['DOMAIN_NAME']}"
opt_trial_params = project_dir / "simulations" / "run_dds" / "settings" / "SUMMA" / "trialParams.nc"
main_trial_params = project_dir / "settings" / "SUMMA" / "trialParams.nc"

# Backup and copy
if opt_trial_params.exists():
    # Backup original
    if main_trial_params.exists():
        backup_path = main_trial_params.with_suffix('.nc.backup')
        shutil.copy(main_trial_params, backup_path)
        print(f"Backed up original to: {backup_path}")
    
    # Copy optimized parameters
    shutil.copy(opt_trial_params, main_trial_params)
    print(f"✅ Copied optimized parameters to: {main_trial_params}")
else:
    print(f"⚠️ Optimized parameters not found: {opt_trial_params}")


## Step 4b: Run model

In [ ]:
# Ensure parameter snapshots are saved inside the experiment folder
experiment_dir = Path(config_dict['CONFLUENCE_DATA_DIR']) / f"domain_{config_updates['DOMAIN_NAME']}" / "simulations" / name
parameters_dir = experiment_dir / "parameters"
parameters_dir.mkdir(parents=True, exist_ok=True)
shutil.copy(localParamInfo_file, parameters_dir / "localParamInfo.txt")
shutil.copy(basinParamInfo_file, parameters_dir / "basinParamInfo.txt")
print(f"Saved parameter files to: {parameters_dir}")

In [ ]:
# =============================================================================
# NOW RUN THE MODEL
# =============================================================================

print("\n" + "="*60)
print("🚀 Running SUMMA model...")
print("="*60)

confluence.managers['model'].run_models()
print("✅ Basin-scale integrated simulation complete")

## Step 5: Streamflow Evaluation and Basin Performance Assessment

In [ ]:
summa_dir = project_dir / "simulations" / config_dict['EXPERIMENT_ID'] / "SUMMA"
sim_ds = xr.open_dataset(list(summa_dir.glob("*_timestep.nc"))[0])
save = False

# Get basin area
shp_path = project_dir / "shapefiles" / "catchment" / "East_River_lumped_HRUs_GRUs.shp"
gdf = gpd.read_file(shp_path).to_crs('EPSG:32611')
basin_area = gdf.geometry.area.sum()

# Calculate discharge
runoff = sim_ds['scalarTotalRunoff'].squeeze()
discharge = runoff * basin_area  # m³/s

# Load observations
obs_path = project_dir / "observations" / "streamflow" / "preprocessed" / "East_River_lumped_streamflow_processed.csv"
obs_df = pd.read_csv(obs_path, parse_dates=['datetime'])
obs_df.set_index('datetime', inplace=True)

# Align to common period (skip first year for spin-up)
start = '2014-10-01'
end = '2022-09-30'
sim_daily = discharge.resample(time='D').mean().sel(time=slice(start, end))
obs_daily = obs_df['discharge_cms'].resample('D').mean().loc[start:end]

# Create figure
fig = plt.figure(figsize=(16, 12))

# 1. Monthly climatology comparison
ax1 = fig.add_subplot(2, 2, 1)
sim_monthly = sim_daily.groupby('time.month').mean()
obs_monthly = obs_daily.groupby(obs_daily.index.month).mean()
months = range(1, 13)
ax1.bar([m-0.2 for m in months], [float(obs_monthly.loc[m]) for m in months], 
        width=0.4, label='Observed', color='blue', alpha=0.7)
ax1.bar([m+0.2 for m in months], [float(sim_monthly.sel(month=m)) for m in months], 
        width=0.4, label='Simulated', color='red', alpha=0.7)
ax1.set_xticks(months)
ax1.set_xticklabels(['J','F','M','A','M','J','J','A','S','O','N','D'])
ax1.set_ylabel('Discharge (m³/s)')
ax1.set_title('Monthly Mean Discharge\nPeak: Obs=May, Sim=April (1 month early)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. Water year 2017 (big snow year) hydrograph
ax2 = fig.add_subplot(2, 2, 2)
wy2017_sim = sim_daily.sel(time=slice('2016-10-01', '2017-09-30'))
wy2017_obs = obs_daily.loc['2016-10-01':'2017-09-30']
ax2.plot(wy2017_obs.index, wy2017_obs.values, 'b-', label='Observed', lw=1.5)
ax2.plot(pd.to_datetime(wy2017_sim.time.values), wy2017_sim.values, 'r-', label='Simulated', lw=1.5, alpha=0.8)
ax2.set_ylabel('Discharge (m³/s)')
ax2.set_title('WY2017 Hydrograph (Big Snow Year)\nNote: Sim peaks ~1 month early')
ax2.legend()
ax2.grid(True, alpha=0.3)

# 3. SWE comparison with East  if available
ax3 = fig.add_subplot(2, 2, 3)
swe = sim_ds['scalarSWE'].squeeze().resample(time='D').mean()
tuolumne_meadows = pd.read_csv("/scratch/dlhogan/ess-project-data/domain_East_River_lumped/observations/snow/TUM_TUOLUMNE_MEADOWS_cdec_obs.csv", parse_dates=['datetime'], index_col='datetime')
tuolumne_meadows_swe = tuolumne_meadows['SWE']*25.4
# filter values 
# filter extreme values
tuolumne_meadows_swe = tuolumne_meadows_swe[(tuolumne_meadows_swe < 2000) & (tuolumne_meadows_swe >= 0)]
tuolumne_meadows_swe_monthly = tuolumne_meadows_swe.groupby(tuolumne_meadows_swe.index.month).mean()
# Plot multi-year average SWE cycle
swe_monthly = swe.sel(time=slice(start, end)).groupby('time.month').mean()
ax3.plot(months, [float(swe_monthly.sel(month=m)) for m in months], 'b-o', lw=2, label='Simulated SWE')
ax3.plot(months, tuolumne_meadows_swe_monthly.values, 'r--o', lw=2, label='Dana  Observed')
ax3.set_xticks(months)
ax3.set_xticklabels(['J','F','M','A','M','J','J','A','S','O','N','D'])
ax3.set_ylabel('SWE (mm)')
ax3.set_title('Mean Monthly SWE\nPeak in March, Melt-out by July')
ax3.grid(True, alpha=0.3)
ax3.legend()

# 4. Timing metrics by year
ax4 = fig.add_subplot(2, 2, 4)
timing_data = []
for year in range(2012, 2022):
    wy_start = f'{year-1}-10-01'
    wy_end = f'{year}-09-30'
    
    sim_wy = sim_daily.sel(time=slice(wy_start, wy_end))
    obs_wy = obs_daily.loc[wy_start:wy_end]
    
    # Find center of mass (timing metric)
    sim_vals = sim_wy.values
    obs_vals = obs_wy.values
    
    days = np.arange(len(sim_vals))
    if np.sum(sim_vals) > 0:
        sim_com = np.sum(days * sim_vals) / np.sum(sim_vals)
    else:
        sim_com = np.nan
        
    obs_com = np.sum(days * obs_vals[~np.isnan(obs_vals)]) / np.sum(obs_vals[~np.isnan(obs_vals)])
    
    timing_data.append({'year': year, 'sim_com': sim_com, 'obs_com': obs_com})

timing_df = pd.DataFrame(timing_data)
ax4.scatter(timing_df['obs_com'], timing_df['sim_com'], s=100, c='blue', edgecolor='black')
ax4.plot([100, 300], [100, 300], 'k--', label='1:1 line')
for _, row in timing_df.iterrows():
    ax4.annotate(f"WY{int(row['year'])}", (row['obs_com']+3, row['sim_com']))
ax4.set_xlabel('Observed Center of Mass (day of WY)')
ax4.set_ylabel('Simulated Center of Mass (day of WY)')
ax4.set_title('Runoff Timing: Center of Mass\nPoints below line = early runoff')
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
if save:
    plt.savefig('/scratch/dlhogan/ess-project-data/domain_East_River_lumped/plots/timing_diagnostic.png', dpi=150)
    print("Saved: timing_diagnostic.png")

# Print summary statistics
print("\n=== TIMING DIAGNOSTICS ===")
print(f"Mean timing offset: {float(timing_df['sim_com'].mean() - timing_df['obs_com'].mean()):.0f} days (negative = early)")
print(f"Peak month: Obs={obs_monthly.idxmax()}, Sim={int(sim_monthly.argmax())+1}")

# Calculate NSE excluding spin-up
common_idx = pd.to_datetime(sim_daily.time.values)
obs_aligned = obs_daily.reindex(common_idx)
valid = ~obs_aligned.isna()
sim_v = sim_daily.values[valid]
obs_v = obs_aligned.values[valid]
nse = 1 - np.sum((obs_v - sim_v)**2) / np.sum((obs_v - np.mean(obs_v))**2)
print(f"NSE (post spin-up): {nse:.3f}")

sim_ds.close()
plt.show()

In [ ]:
# Load observed streamflow data
obs_path = project_dir / "observations" / "streamflow" / "preprocessed" / f"{config_dict['DOMAIN_NAME']}_streamflow_processed.csv"
obs_df = pd.read_csv(obs_path, parse_dates=['datetime'])
obs_df.set_index('datetime', inplace=True)
    
# Load simulated streamflow from mizuRoute
summa_dir = project_dir / "simulations" / config_dict['EXPERIMENT_ID'] / "SUMMA"
routing_dir = project_dir / "simulations" / config_dict['EXPERIMENT_ID'] / "mizuRoute"
routing_files = list(routing_dir.glob("*.nc"))

if routing_files:
    # Load mizuRoute output
    routing_ds = xr.open_dataset(routing_files[0])
    
    # Extract streamflow variable (typically IRFroutedRunoff)
    
    sim_streamflow = routing_ds['IRFroutedRunoff']
    sim_df = sim_streamflow.to_pandas()    
    routing_ds.close()

else:
    sim_ds = xr.open_dataset(list(summa_dir.glob("*_timestep.nc"))[0])
    shp_file = gpd.read_file(str(project_dir / "shapefiles" / "catchment" / f"{config_dict['DOMAIN_NAME']}_HRUs_{config_dict['DOMAIN_DISCRETIZATION']}.shp"))
    shp_area = shp_file['GRU_area'].values[1]
    sim_ds['scalarTotalRunoff'] = sim_ds['scalarTotalRunoff'] * shp_area
    sim_streamflow = sim_ds['scalarTotalRunoff']    
    sim_df = sim_streamflow.to_pandas()
    # Calculate discharge
    
    sim_ds.close()

# =============================================================================
# STREAMFLOW PERFORMANCE EVALUATION
# =============================================================================

print(f"\n Streamflow Performance Assessment...")

# Align data to common period
start_date = max(obs_df.index.min(), sim_df.index.min())
end_date = min(obs_df.index.max(), sim_df.index.max())

# Skip initial spinup period
start_date = start_date + pd.DateOffset(months=6)

print(f"   Evaluation period: {start_date} to {end_date}")
print(f"   Duration: {(end_date - start_date).days} days")

# Filter to common period and resample to daily
obs_daily = obs_df['discharge_cms'].resample('D').mean().loc[start_date:end_date]
sim_daily = sim_df.resample('D').mean().loc[start_date:end_date]

# Ensure sim_daily is a Series (in case it's a DataFrame with one column)
if isinstance(sim_daily, pd.DataFrame):
    if sim_daily.shape[1] == 1:
        sim_daily = sim_daily.iloc[:, 0]  # Take the first (and likely only) column
    else:
        print(f"⚠️  sim_daily has {sim_daily.shape[1]} columns. Using the first column.")
        print(f"   Available columns: {list(sim_daily.columns)}")
        sim_daily = sim_daily.iloc[:, 0]

# Remove any remaining NaN values
valid_mask = ~(obs_daily.isna() | sim_daily.isna())
obs_valid = obs_daily[valid_mask]
sim_valid = sim_daily[valid_mask]    
print(f"   Valid paired observations: {len(obs_valid)} days")

# Calculate comprehensive performance metrics
print(f"\n📈 Streamflow Performance Metrics:")

# Basic statistics
rmse = np.sqrt(((obs_valid - sim_valid) ** 2).mean())
bias = (sim_valid - obs_valid).mean()
mae = np.abs(obs_valid - sim_valid).mean()

# Relative metrics
pbias = 100 * bias / obs_valid.mean()

# Nash-Sutcliffe Efficiency
nse = 1 - ((obs_valid - sim_valid) ** 2).sum() / ((obs_valid - obs_valid.mean()) ** 2).sum()

# Kling-Gupta Efficiency  
r = obs_valid.corr(sim_valid)
alpha = sim_valid.std() / obs_valid.std()
beta = sim_valid.mean() / obs_valid.mean()
kge = 1 - np.sqrt((r - 1)**2 + (alpha - 1)**2 + (beta - 1)**2)

# Display performance metrics
print(f"   📊 RMSE: {rmse:.2f} m³/s")
print(f"   📊 Bias: {bias:+.2f} m³/s ({pbias:+.1f}%)")
print(f"   📊 MAE: {mae:.2f} m³/s")
print(f"   📊 Correlation (r): {r:.3f}")
print(f"   📊 Nash-Sutcliffe (NSE): {nse:.3f}")
print(f"   📊 Kling-Gupta (KGE): {kge:.3f}")

# Hydrologic signature analysis
print(f"\n🌊 Hydrologic Signature Analysis:")

# Flow statistics
obs_q95 = obs_valid.quantile(0.95)  # High flows
sim_q95 = sim_valid.quantile(0.95)
obs_q05 = obs_valid.quantile(0.05)  # Low flows  
sim_q05 = sim_valid.quantile(0.05)

print(f"   High flows (Q95): Obs={obs_q95:.1f}, Sim={sim_q95:.1f} m³/s")
print(f"   Low flows (Q05): Obs={obs_q05:.1f}, Sim={sim_q05:.1f} m³/s")

# Seasonal timing
obs_monthly = obs_valid.groupby(obs_valid.index.month).mean()
sim_monthly = sim_valid.groupby(sim_valid.index.month).mean()
peak_month_obs = obs_monthly.idxmax()
peak_month_sim = sim_monthly.idxmax()

print(f"   Peak flow timing: Obs=Month {peak_month_obs}, Sim=Month {peak_month_sim}")

# =============================================================================
# STREAMFLOW VISUALIZATION
# =============================================================================

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Time series comparison (top left)
ax1 = axes[0, 0]
ax1.plot(obs_valid.index, obs_valid.values, 'b-', 
         label='USGS Observed', linewidth=1.5, alpha=0.8)
ax1.plot(sim_valid.index, sim_valid.values, 'r-', 
         label='SUMMA', linewidth=1.5, alpha=0.8)

ax1.set_ylabel('Discharge (m³/s)', fontsize=11)
ax1.set_title('Streamflow Time Series', fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Add performance metrics
metrics_text = f'NSE: {nse:.3f}\nKGE: {kge:.3f}\nBias: {pbias:+.1f}%'
ax1.text(0.02, 0.95, metrics_text, transform=ax1.transAxes,
         bbox=dict(facecolor='white', alpha=0.8), fontsize=10, verticalalignment='top')

# Scatter plot (top right)
ax2 = axes[0, 1]
ax2.scatter(obs_valid, sim_valid, alpha=0.5, c='blue', s=20)
max_val = max(obs_valid.max(), sim_valid.max())
ax2.plot([0, max_val], [0, max_val], 'k--', label='1:1 line')
ax2.set_xlabel('Observed (m³/s)', fontsize=11)
ax2.set_ylabel('Simulated (m³/s)', fontsize=11)
ax2.set_title('Obs vs Sim Streamflow', fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Monthly climatology (bottom left)
ax3 = axes[1, 0]
months = range(1, 13)
month_names = ['J', 'F', 'M', 'A', 'M', 'J', 'J', 'A', 'S', 'O', 'N', 'D']

ax3.plot(months, obs_monthly.values, 'o-', label='Observed', 
         color='blue', linewidth=2, markersize=6)
ax3.plot(months, sim_monthly.values, 'o-', label='Simulated', 
         color='red', linewidth=2, markersize=6)

ax3.set_xticks(months)
ax3.set_xticklabels(month_names)
ax3.set_ylabel('Mean Discharge (m³/s)', fontsize=11)
ax3.set_title('Seasonal Flow Regime', fontweight='bold')
ax3.legend()
ax3.grid(True, alpha=0.3)

# Flow duration curve (bottom right)
ax4 = axes[1, 1]

# Calculate exceedance probabilities
obs_sorted = obs_valid.sort_values(ascending=False)
sim_sorted = sim_valid.sort_values(ascending=False)
obs_ranks = np.arange(1., len(obs_sorted) + 1) / len(obs_sorted) * 100
sim_ranks = np.arange(1., len(sim_sorted) + 1) / len(sim_sorted) * 100

ax4.semilogy(obs_ranks, obs_sorted, 'b-', label='Observed', linewidth=2)
ax4.semilogy(sim_ranks, sim_sorted, 'r-', label='Simulated', linewidth=2)

ax4.set_xlabel('Exceedance Probability (%)', fontsize=11)
ax4.set_ylabel('Discharge (m³/s)', fontsize=11)
ax4.set_title('Flow Duration Curve', fontweight='bold')
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.suptitle(f'Basin-Scale Streamflow Evaluation - {config_dict["DOMAIN_NAME"]}',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Water Balance Components

In [ ]:
# =============================================================================
# WATER BALANCE COMPONENT TIMESERIES - MONTHLY TOTALS
# =============================================================================
import matplotlib.dates as mdates

# Load SUMMA output for water balance analysis
summa_file = list(summa_dir.glob("*.nc"))
if summa_file:
    ds_wb = xr.open_dataset(summa_file[0])
    ds_wb = ds_wb.sel(time=slice('2014-01-01', '2022-09-30'))  # Focus on common period
    
    # Extract time and convert to pandas datetime
    time_wb = pd.to_datetime(ds_wb['time'].values)
    
    # Initialize monthly series
    monthly_precip = None
    monthly_swe = None
    monthly_delta_sm = None
    monthly_et = None
    monthly_runoff = None
    monthly_baseflow = None
    
    # 1. Precipitation (convert from kg/m2/s to mm/month via daily totals)
    if 'pptrate' in ds_wb:
        precip = ds_wb['pptrate'].values.squeeze() * 86400 / 24  # kg/m2/s to mm/day
        precip_daily = pd.Series(precip, index=time_wb)
        monthly_precip = precip_daily.resample('MS').sum()  # Monthly total
    
    # 2. SWE (keep as daily mean for end-of-month values, kg/m2 = mm)
    if 'scalarSWE' in ds_wb:
        swe_wb = ds_wb['scalarSWE'].values.squeeze()
        swe_daily = pd.Series(swe_wb, index=time_wb)
        monthly_swe = swe_daily.resample('MS').mean()  # Monthly mean for visualization
    
    # 3. Soil Moisture Change (Delta SM in mm/month)
    if 'scalarTotalSoilLiq' in ds_wb:
        soil_liq = ds_wb['scalarTotalSoilLiq'].values.squeeze()  # kg/m2 = mm
        soil_daily = pd.Series(soil_liq, index=time_wb)
        # Calculate monthly change: SM_end - SM_start for each month
        monthly_delta_sm = soil_daily.resample('MS').last() - soil_daily.resample('MS').first()
    
    # 4. Evapotranspiration (convert from W/m2 to mm/month)
    latent_heat_vap = 2.45e6  # J/kg
    if 'scalarLatHeatTotal' in ds_wb:
        et_wm2 = ds_wb['scalarLatHeatTotal'].values.squeeze()  # W m-2
        et_mm_day = et_wm2 * 86400 / 24 / latent_heat_vap
        et_daily = pd.Series(et_mm_day, index=time_wb)
        monthly_et = et_daily.resample('MS').sum()  # Monthly total
    
    # 5. Runoff (convert from m/s to mm/month)
    if 'scalarTotalRunoff' in ds_wb:
        runoff_wb = ds_wb['scalarTotalRunoff'].values.squeeze() * 86400 / 24 * 1000  # m/s to mm/day
        runoff_daily = pd.Series(runoff_wb, index=time_wb)
        monthly_runoff = runoff_daily.resample('MS').sum()  # Monthly total
    
    # 6. Aquifer/Baseflow (convert from m/s to mm/month)
    if 'scalarAquiferBaseflow' in ds_wb:
        baseflow = ds_wb['scalarAquiferBaseflow'].values.squeeze() * 86400 / 24 * 1000  # m/s to mm/day
        baseflow_daily = pd.Series(baseflow, index=time_wb)
        monthly_baseflow = baseflow_daily.resample('MS').sum()  # Monthly total
    elif 'scalarSoilBaseflow' in ds_wb:
        baseflow = ds_wb['scalarSoilBaseflow'].values.squeeze() * 86400 / 24 * 1000
        baseflow_daily = pd.Series(baseflow, index=time_wb)
        monthly_baseflow = baseflow_daily.resample('MS').sum()  # Monthly total
    
    # =============================================================================
    # VISUALIZATION: PANEL PLOTS
    # =============================================================================
    
    fig, axes = plt.subplots(3, 2, figsize=(16, 12), sharex=True)
    axes = axes.flatten()
    
    # 1. Precipitation
    ax = axes[0]
    if monthly_precip is not None:
        ax.bar(monthly_precip.index, monthly_precip.values, width=25, color='steelblue', alpha=0.7, label='Precipitation')
        ax.set_ylabel('mm/month', fontsize=10)
        ax.set_title('Precipitation', fontweight='bold')
        ax.legend(loc='upper right')
        ax.grid(True, alpha=0.3)
    
    # 2. SWE (daily mean)
    ax = axes[1]
    if monthly_swe is not None:
        ax.fill_between(monthly_swe.index, monthly_swe.values, color='lightblue', alpha=0.7)
        ax.plot(monthly_swe.index, monthly_swe.values, color='blue', linewidth=1.5, label='SWE')
        ax.set_ylabel('mm', fontsize=10)
        ax.set_title('Snow Water Equivalent (Monthly Mean)', fontweight='bold')
        ax.legend(loc='upper right')
        ax.grid(True, alpha=0.3)
    
    # 3. Soil Moisture Change (Delta SM)
    ax = axes[2]
    if monthly_delta_sm is not None:
        colors = ['red' if x < 0 else 'saddlebrown' for x in monthly_delta_sm.values]
        ax.bar(monthly_delta_sm.index, monthly_delta_sm.values, width=25, color=colors, alpha=0.6, label='ΔSM')
        ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
        ax.set_ylabel('ΔSM (mm/month)', fontsize=10)
        ax.set_title('Soil Moisture Change (positive = wetting)', fontweight='bold')
        ax.legend(loc='upper right')
        ax.grid(True, alpha=0.3)
    
    # 4. Evapotranspiration
    ax = axes[3]
    if monthly_et is not None:
        ax.bar(monthly_et.index, (-monthly_et).values.clip(min=0), width=25, color='green', alpha=0.6, label='Total ET')
        ax.set_ylabel('mm/month', fontsize=10)
        ax.set_title('Evapotranspiration', fontweight='bold')
        ax.legend(loc='upper right')
        ax.grid(True, alpha=0.3)
    
    # 5. Runoff
    ax = axes[4]
    if monthly_runoff is not None:
        ax.bar(monthly_runoff.index, monthly_runoff.values, width=25, color='coral', alpha=0.6, label='Total Runoff')
        ax.set_ylabel('mm/month', fontsize=10)
        ax.set_title('Total Runoff', fontweight='bold')
        ax.legend(loc='upper right')
        ax.grid(True, alpha=0.3)
    
    # 6. Baseflow
    ax = axes[5]
    if monthly_baseflow is not None:
        ax.bar(monthly_baseflow.index, monthly_baseflow.values, width=25, color='purple', alpha=0.6, label='Baseflow')
        ax.set_ylabel('mm/month', fontsize=10)
        ax.set_title('Baseflow', fontweight='bold')
        ax.legend(loc='upper right')
        ax.grid(True, alpha=0.3)
    
    # Format x-axis
    for ax in axes:
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
        ax.xaxis.set_major_locator(mdates.YearLocator())
    
    plt.suptitle(f'Water Balance Components (Monthly) - {config_dict["DOMAIN_NAME"]}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
        # =============================================================================
    # WATER BALANCE TIME SERIES - ANNUAL CYCLE (Monthly averages across all years)
    # =============================================================================
    
    fig, ax = plt.subplots(figsize=(14, 6))
    
    # Calculate monthly averages across all water years
    months = range(1, 13)
    month_names = ['Oct', 'Nov', 'Dec', 'Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep']
    
    # Group by water year month (Oct=1, Nov=2, ..., Sep=12)
    monthly_precip_cycle = monthly_precip.groupby(monthly_precip.index.month).mean()
    monthly_et_cycle = monthly_et.groupby(monthly_et.index.month).mean()
    monthly_runoff_cycle = monthly_runoff.groupby(monthly_runoff.index.month).mean()
    monthly_delta_sm_cycle = monthly_delta_sm.groupby(monthly_delta_sm.index.month).mean()
    monthly_baseflow_cycle = monthly_baseflow.groupby(monthly_baseflow.index.month).mean()
    
    
    # Reorder to water year (Oct-Sep instead of Jan-Dec)
    wy_order = [10, 11, 12, 1, 2, 3, 4, 5, 6, 7, 8, 9]  # Water year months
    wy_labels = ['Oct', 'Nov', 'Dec', 'Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep']
    
    # Reorder the data
    if monthly_precip is not None:
        precip_wy = [monthly_precip_cycle.get(m, np.nan) for m in wy_order]
        ax.plot(range(12), precip_wy, 'o-', label='P (Precipitation)', 
                color='steelblue', linewidth=2.5, markersize=6)
    
    if monthly_et is not None:
        et_wy = [monthly_et_cycle.get(m, np.nan) for m in wy_order]
        ax.plot(range(12), et_wy, 'o-', label='-ET (Evapotranspiration)', 
                color='green', linewidth=2.5, markersize=6)
    
    if monthly_runoff is not None:
        runoff_wy = [-monthly_runoff_cycle.get(m, np.nan) for m in wy_order]
        ax.plot(range(12), runoff_wy, 'o-', label='-Q (Runoff)', 
                color='coral', linewidth=2.5, markersize=6)
    
    if monthly_delta_sm is not None:
        sm_wy = [monthly_delta_sm_cycle.get(m, np.nan) for m in wy_order]
        ax.plot(range(12), sm_wy, 'o-', label='ΔSM (Soil Moisture Change)', 
                color='brown', linewidth=2.5, markersize=6)
    
    ax.axhline(0, color='black', linewidth=1, linestyle='--', alpha=0.5)
    ax.set_xticks(range(12))
    ax.set_xticklabels(wy_labels, fontsize=11)
    ax.set_ylabel('Water Flux (mm/month)', fontsize=12)
    ax.set_xlabel('Water Year Month', fontsize=12)
    ax.set_title('Annual Water Balance Cycle\nAverage monthly fluxes across all water years: P - ET - Q ± ΔSM ≈ 0', 
                 fontweight='bold', fontsize=13)
    ax.legend(loc='upper right', fontsize=11)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # =============================================================================
    # ANNUAL WATER BALANCE SUMMARY
    # =============================================================================
    
    print("\n" + "=" * 80)
    print("ANNUAL WATER BALANCE SUMMARY (mm/year)")
    print("=" * 80)
    
    # Calculate annual totals
    annual_summary = []
    years = monthly_precip.index.year.unique()
    for year in years:
        year_mask = monthly_precip.index.year == year
        annual = {
            'Year': year,
            'P (mm)': monthly_precip[year_mask].sum() if monthly_precip is not None else np.nan,
            'ET (mm)': monthly_et[year_mask].sum() if monthly_et is not None else np.nan,
            'Q (mm)': monthly_runoff[year_mask].sum() if monthly_runoff is not None else np.nan,
            'ΔSM (mm)': monthly_delta_sm[year_mask].sum() if monthly_delta_sm is not None else np.nan,
            'Peak SWE (mm)': monthly_swe[year_mask].max() if monthly_swe is not None else np.nan,
        }
        annual_summary.append(annual)
    
    annual_df = pd.DataFrame(annual_summary)
    print(annual_df.to_string(index=False))
    
    # Water balance closure
    print("\n" + "-" * 80)
    print(f"Mean Annual P:   {annual_df['P (mm)'].mean():7.1f} mm")
    print(f"Mean Annual ET:  {annual_df['ET (mm)'].mean():7.1f} mm")
    print(f"Mean Annual Q:   {annual_df['Q (mm)'].mean():7.1f} mm")
    print(f"Mean Annual ΔSM: {annual_df['ΔSM (mm)'].mean():7.1f} mm")
    
    # Closure check
    imbalance = annual_df['P (mm)'].mean() + annual_df['ET (mm)'].mean() - annual_df['Q (mm)'].mean() - annual_df['ΔSM (mm)'].mean()
    print(f"\nWater Balance Closure: P - ET - Q - ΔSM = {imbalance:.1f} mm/year")
    print("(Residual represents other storage changes: SWE, aquifer, canopy, etc.)")
    
    ds_wb.close()
else:
    print("No SUMMA output file found!")

## ET evaluation

In [ ]:
sim_ds = xr.open_dataset(list(summa_dir.glob("*_timestep.nc"))[0])
open_et = pd.read_csv("/scratch/dlhogan/ess-project-data/domain_East_River_lumped/observations/et/openet_et_ensemble_tuolumne_monthly.csv", index_col=0, parse_dates=True)

In [ ]:
et = sim_ds['scalarLatHeatTotal'].resample(time='ME').mean()

# convert to W/m2 to mm/month
latent_heat = -et * 30 * 24 * 3600 / (2.45e6)  # mm/month
fig, ax = plt.subplots(figsize=(10, 6))
latent_heat.plot(ax=ax, label='SUMMA ET', marker='o')
open_et['et'].plot(ax=ax, label='OpenET Ensemble', marker='s')
ax.set_ylabel('ET (mm/month)', fontsize=12)
ax.set_title('Monthly Evapotranspiration Comparison', fontweight='bold', fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)

## Temperature Evaluation

In [ ]:
tuolumne_meadows = pd.read_csv("/scratch/dlhogan/ess-project-data/domain_East_River_lumped/observations/snow/TUM_TUOLUMNE_MEADOWS_cdec_obs.csv", parse_dates=['datetime'], index_col='datetime')
tuolumne_meadows = tuolumne_meadows.loc['2019-01-01':'2022-01-01']

In [ ]:
tuolumne_meadows_temp =((tuolumne_meadows['AIR TEMP'] - 32) * 5/9)
tuolumne_meadows_temp = tuolumne_meadows_temp[(tuolumne_meadows_temp > -50) & (tuolumne_meadows_temp < 50)]
tuolumne_meadows_temp_monthly = tuolumne_meadows_temp.groupby(tuolumne_meadows_temp.index.month).mean()

sim_ds_airtemp = sim_ds['airtemp'].resample(time='D').mean().sel(time=slice('2019-01-01','2021-12-31')).squeeze() - 273.16
# plot timeseries
fig, axs = plt.subplots(ncols=1, nrows=2, tight_layout=True, figsize=(6,8))
axs[0].scatter(tuolumne_meadows_temp.values, sim_ds_airtemp.values, alpha=0.5, ec='k')
axs[0].plot([-20,20], [-20,20], 'k--', label='1:1 line')
axs[0].legend()
axs[0].set_xlabel('TUOLUMNE MEADOWS')
axs[0].set_ylabel('SUMMA')
axs[0].set_title("SWE Time Series Comparison")

# plot monthly average comparison
tuolumne_meadows_temp_monthly.plot(ax=axs[1], label='TUOLUMNE MEADOWS')
sim_ds_airtemp.groupby('time.month').mean().plot(ax=axs[1], label='SUMMA')
axs[1].legend()
axs[1].set_xlabel('Month')
axs[1].set_title("SWE Monthly Average Comparison")

print(f"Monthly bias (SUMMA - TUOLUMNE MEADOWS): {(sim_ds_airtemp.groupby('time.month').mean() - tuolumne_meadows_temp_monthly).values}")

## SWE evaluation

In [ ]:
tuolumne_meadows_swe = tuolumne_meadows['SWE']*25.4
# filter extreme values
tuolumne_meadows_swe = tuolumne_meadows_swe[tuolumne_meadows_swe < 2000]
tuolumne_meadows_swe_monthly = tuolumne_meadows_swe.groupby(tuolumne_meadows_swe.index.month).mean()
# plot timeseries
fig, axs = plt.subplots(ncols=1, nrows=2, tight_layout=True, figsize=(6,6))
tuolumne_meadows_swe.plot(ax=axs[0], label='TUOLUMNE MEADOWS')
sim_ds['scalarSWE'].plot(ax=axs[0], label='SUMMA')
axs[0].legend()
axs[0].set_xlabel('Date')
axs[0].set_title("SWE Time Series Comparison")

# plot monthly average comparison
tuolumne_meadows_swe_monthly.plot(ax=axs[1], label='TUOLUMNE MEADOWS')
sim_ds['scalarSWE'].groupby('time.month').mean().plot(ax=axs[1], label='SUMMA')
axs[1].legend()
axs[1].set_xlabel('Month')
axs[1].set_title("SWE Monthly Average Comparison")

In [ ]:
tuolumne_meadows_ppt = (tuolumne_meadows['PRECIPITATION']*25.4)
tuolumne_meadows_ppt = tuolumne_meadows_ppt[tuolumne_meadows_ppt < 500]
fig, ax = plt.subplots()
# ax.scatter(tuolumne_meadows_ppt.fillna(0).values, (sim_ds['pptrate']*3600).resample(time='1D').sum().sel(time=tuolumne_meadows_ppt.index.date).values, alpha=0.5)
tuolumne_meadows_ppt.cumsum().plot(ax=ax, label='tuolumne MEADOWS')
(sim_ds['pptrate']*3600).resample(time='1D').sum().cumsum().plot(ax=ax, label='SUMMA')
ax.legend()

## Summary: Lumped Basin-Scale Streamflow Simulation
This tutorial successfully demonstrated the critical scaling transition from point-scale process validation to basin-scale integrated streamflow simulation using CONFLUENCE. Through the East River at Almont case study, we illustrated how the same standardized workflow framework seamlessly scales from individual site validation to watershed-scale prediction while maintaining scientific rigor and computational efficiency.

## Key Methodological Achievements
The tutorial established watershed-scale modeling capabilities by successfully transitioning from point measurements to integrated basin response through automated watershed delineation and lumped representation strategies. Spatial aggregation techniques were demonstrated through catchment-averaged characteristic development that captures essential watershed properties while maintaining computational tractability. Integrated process simulation was achieved by coupling SUMMA's process-based physics with mizuRoute streamflow routing to transform distributed runoff generation into streamflow predictions at the basin outlet.

## Scientific Process Understanding
The evaluation demonstrated CONFLUENCE's ability to simulate integrated watershed response through the successful representation of snow accumulation, melt, and runoff generation processes across elevation gradients in a mountain environment. Seasonal flow regime capture was validated through comparison with long-term Water Survey of Canada observations, showing the model's capability to represent pronounced spring freshet patterns and low-flow periods. Water balance closure was maintained through conservation of mass principles while scaling from point processes to basin-integrated streamflow generation.

## Framework Scalability Validation
This tutorial confirmed CONFLUENCE's seamless scaling capabilities by applying identical workflow principles from point validation through basin-scale prediction without requiring fundamental architectural changes. The model-agnostic preprocessing approach proved equally effective for basin-averaged forcing and validation data preparation, reinforcing the framework's broad applicability. Computational efficiency was demonstrated through lumped representation strategies that enable rapid execution suitable for calibration, uncertainty analysis, and operational applications while maintaining process-based physical realism.

This foundation in lumped basin modeling establishes the essential principles for understanding watershed-scale hydrological behavior and 
prepares for the spatial complexity introduced in semi-distributed and fully distributed modeling approaches in subsequent tutorials.

### Next Focus: Semi-Distributed Watershed Modelling 

**Ready to explore Semi-Distributed basin simulations?** → **[Tutorial 02b: Basin Scale - Semi-Distributed Watershed](./02b_basin_semi_distributed.ipynb)**